In [1]:
# import packages
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
import glob
import cv2
import zipfile
import sys


In [2]:
# package versions
print('numpy version is:{}'.format(np.__version__))
print('pytorch version is:{}'.format(torch.__version__))
print('opencv version is:{}'.format(cv2.__version__))

numpy version is:2.0.2
pytorch version is:2.9.0+cu126
opencv version is:4.12.0


In [3]:
# connect to google drive
from google.colab import drive, userdata
drive.mount('/content/drive/')



Mounted at /content/drive/


In [4]:
#add folder holding autoencoder class to sys path
project_path= userdata.get('cnn_folder')

if project_path not in sys.path:
  sys.path.append(project_path)

from model_folder.model_autoenc import autoencoder

In [5]:
# extract images from zip folder
zip1= userdata.get('zip_folder')

with zipfile.ZipFile(zip1, mode= 'r') as zip2:
  zip2.extractall()

In [6]:
# make custom dataset
class custom_dataset(Dataset):
  def __init__(self, img_dir, num_img, transform= None):
    self.img_dir= img_dir
    self.transform= transform
    self.num_img= num_img

  def __len__(self):
    img_str= self.img_dir + '/*png'
    # image list
    img_list= glob.glob(img_str)
    img_list1= img_list[:self.num_img]
    return len(img_list1)

  def __getitem__(self, idx):
    img_str= self.img_dir + '/*png'
    img_list= glob.glob(img_str)
    #read image
    img0= cv2.imread(img_list[idx])
    # convert to hsv
    img_hsv= cv2.cvtColor(img0, cv2.COLOR_BGR2HSV)
    # keep saturation channel
    s_norm= img_hsv[:, :, 1]/255.0
    # normalize saturation
    sat1= s_norm.reshape(s_norm.shape[0], s_norm.shape[1], 1)
    sat1= sat1.astype(np.float32)
    if self.transform is not None:
      sat1= self.transform(sat1)
    return sat1


In [7]:
# define transform- convert to torch tensor then resize to 224 x 224
img_transform= transforms.Compose([transforms.ToTensor(), transforms.Resize((224, 224))])

train_healthy_dir= userdata.get('train_healthy')

# make dataset
all_dataset= custom_dataset(img_dir= train_healthy_dir, num_img= 5000, transform= img_transform)

# split dataset into training and validation
train_dataset, val_dataset= random_split(all_dataset, lengths= [0.8, 0.2], generator= torch.Generator().manual_seed(24))

# check size of datasets post split
print('length of train dataset is:{}'.format(len(train_dataset)))
print('length of val dataset is:{}'.format(len(val_dataset)))

length of train dataset is:4000
length of val dataset is:1000


In [8]:
# make the dataloaders
# batch size
batch1= 25

train_dataloader= DataLoader(train_dataset, batch_size= batch1)
val_dataloader= DataLoader(val_dataset, batch_size= batch1)

# check size of dataloader
for i in train_dataloader:
  print('shape of train dataloader is:{}'.format(i.shape))
  for k in range(batch1):
    print(i[k].shape)
    break
  break

# check size of dataloader
for j in val_dataloader:
  print('shape of val dataloader is:{}'.format(j.shape))
  break

shape of train dataloader is:torch.Size([25, 1, 224, 224])
torch.Size([1, 224, 224])
shape of val dataloader is:torch.Size([25, 1, 224, 224])


In [9]:
# make the early stopping class

class early_stop():
  def __init__(self, patience, delta, mode= 'min', verbose= True):
    self.patience= patience
    self.delta= delta
    self.mode= mode
    self.verbose= verbose
    self.best_loss= None
    self.stop_training= False
    self.counter= 0

# function to stop training if val loss doesnt improve
  def check_loss(self, val_loss):
    if self.best_loss is None or (self.best_loss - self.delta) > val_loss:
      self.counter= 0
      self.best_loss= val_loss
    else:
      self.counter= self.counter + 1
      if self.counter >= self.patience:
        self.stop_training= True
        print('Training ended due to no significant change in val_loss')

In [10]:
# setup model
# use gpu if available
device= torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# send model to gpu if available
model= autoencoder().to(device)
optimizer= torch.optim.Adam(model.parameters(), lr= 0.001)
early_stopping= early_stop(patience= 5, delta= 0.00005, mode= 'min', verbose= True)
mse1= nn.MSELoss()
# save model
save_str= userdata.get('save_path')


In [11]:
# setup training and validation loops

epoch1= 50
train_loss= 0.0
train_list, val_list= [], []
val_loss= 0.0
for i in range(epoch1):
  # training loop
  model.train()
  for j in train_dataloader:
    j= j.to(device)
    recon= model(j)
    loss= mse1(j, recon)
    total_train_loss= train_loss + loss.item()
    # clear previous
    optimizer.zero_grad()
    # update weights
    loss.backward()
    # advance optimizer
    optimizer.step()
  avg_train_loss= total_train_loss / len(train_dataloader)
  train_list.append(avg_train_loss)
  print('Epoch {} with training loss= {}'.format(i+1, avg_train_loss))

# validation loop
  model.eval()
  with torch.no_grad():
    for k in val_dataloader:
      k= k.to(device)
      val_recon= model(k)
      loss1= mse1(k, val_recon)
      total_val_loss= val_loss + loss1.item()
    avg_val_loss= total_val_loss / len(val_dataloader)
    val_list.append(avg_val_loss)
    # check early stopping
    early_stopping.check_loss(val_loss= avg_val_loss)
    if early_stopping.stop_training:
      print('Training ended at epoch {} due to no improvement in validation loss'.format(i+1))
      break
    # save model
    torch.save(model.state_dict(), save_str)
    print('Epoch {} with validation loss= {}'.format(i+1, avg_val_loss))



Epoch 1 with training loss= 0.00021807414013892413
Epoch 1 with validation loss= 0.0012139009311795234
Epoch 2 with training loss= 0.00021807414013892413
Epoch 2 with validation loss= 0.0012139009311795234
Epoch 3 with training loss= 0.00021807414013892413
Epoch 3 with validation loss= 0.0012139009311795234
Epoch 4 with training loss= 0.00021807414013892413
Epoch 4 with validation loss= 0.0012139009311795234
Epoch 5 with training loss= 0.00021807414013892413
Epoch 5 with validation loss= 0.0012139009311795234
Epoch 6 with training loss= 0.00021807414013892413
Training ended due to no significant change in val_loss
Training ended at epoch 6 due to no improvement in validation loss
